# Paper 3 - 02b . Stage 4 augmentation (cross-lingual + over-refusal counter)

Builds the two augmentation streams over the per-anchor core preference
set produced by notebook 02 (Stage 1+2). Outputs land at
`data/preferences/<short>/preferences_v2.jsonl` (core + augmentations).

Two streams (PAPER3_PLAN section 5.2):

1. **Cross-lingual harmful pairs** (~200 per anchor). Source: HarmBench
   EN prompts, stratified across the 7 standard categories. The model
   sees the EN harmful prompt during DPO training; chosen is a Claude
   Opus 4.7 RO refusal under `TEACHER_SYSTEM_HARMFUL`; rejected is the
   base evaluatee's own EN compliance, mechanically Google-translated
   to RO (Paper 2 R8 lesson: never LLM-MT, the LLM-MT path refuses too
   often on harmful content).

2. **Over-refusal counter-pairs** (~100 per anchor). Source: Paper 2's
   `overrefusal.jsonl`. Chosen is Llama-3.3-70B helpful answer under
   `TEACHER_SYSTEM_BENIGN`. Rejected is sampled from a 15-template
   hand-crafted Romanian apologetic-refusal pool. Templated rather than
   elicited because (a) the rejected side only needs to be recognisable
   as a refusal during DPO, not authentic; (b) elicitation introduces
   per-anchor regeneration cost; (c) some over-refusal prompts are
   borderline (medical first aid, dangerous mushrooms) where an
   elicited refusal might be the *correct* response.

Both streams cache per-pair JSONL on Drive (resumable across sessions);
final preferences_v2.jsonl is rebuilt on every run from the union of
the core preferences_v1.jsonl and the augmentation streams.

**Cost per anchor**: ~$5 (200 Claude calls for cross-lingual chosen +
~100 Llama-70B calls for over-refusal helpful answers + ~200 Google
Translate calls free).

**Pre-flight**: same smoke-gate verification as notebook 02. Stage 1+2
must already have produced a `preferences_v1.jsonl` for the anchor.

In [23]:
%%capture
!pip install -U \
    'requests>=2.32' \
    'tenacity>=9.0' \
    'deep-translator>=1.11' \
    'transformers>=4.51' \
    'accelerate>=1.1' \
    huggingface_hub ipywidgets pyyaml -q

In [24]:
import os, json, gc, time, hashlib, sys
from pathlib import Path
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
import torch

from google.colab import drive
drive.mount("/content/drive")

# --- A100 sanity (cross-lingual stream needs the base anchor for sampling) ---
assert torch.cuda.is_available(), (
    'GPU not available. The cross-lingual stream samples from the base '
    'anchor at temperature 1, which needs an A100 GPU runtime. Switch '
    'Runtime -> Change runtime type -> A100 GPU and re-run.'
)
DEVICE_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {DEVICE_NAME}   VRAM: {VRAM_GB:.1f} GB   torch={torch.__version__}")

torch.set_float32_matmul_precision("high")
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
torch.backends.cudnn.benchmark  = True

# --- Paths ---
DRIVE_ROOT  = Path("/content/drive/MyDrive/PhD/paper3-alignment")
PAPER2_ROOT = Path("/content/drive/MyDrive/PhD/paper2-benchmark")

PREFS_DIR    = DRIVE_ROOT / "data" / "preferences"
AUG_DIR      = DRIVE_ROOT / "data" / "augmentation"
LOGS_DIR     = DRIVE_ROOT / "logs"
for d in [PREFS_DIR, AUG_DIR, LOGS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

sys.path.insert(0, str(PAPER2_ROOT / "src"))

# --- src.prompts: read from Drive (single source of truth) ---
PROMPTS_SRC_DIR = DRIVE_ROOT / "src"
if not (PROMPTS_SRC_DIR / "prompts.py").exists():
    raise RuntimeError(
        f"src/prompts.py not found at {PROMPTS_SRC_DIR / 'prompts.py'}.\n"
        "Copy it from your local rosafety-align/src/prompts.py to that "
        "path in Drive, then re-run this cell."
    )
sys.path.insert(0, str(PROMPTS_SRC_DIR))
from prompts import (
    TEACHER_SYSTEM_HARMFUL, TEACHER_SYSTEM_BENIGN,
    JUDGE_SYSTEM, JUDGE_USER_TEMPLATE,
    PROMPT_DIGESTS, CACHE_NAMESPACE_VERSION,
    sha16, short_of, family_of,
    verify_smoke_gate, display_locked_state, SmokeGateNotPassed,
)

# --- Secrets ---
try:
    from google.colab import userdata
    if not os.environ.get("OPENROUTER_API_KEY"):
        os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY") or ""
    if not os.environ.get("HF_TOKEN"):
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN") or ""
    if os.environ.get("HF_TOKEN"):
        from huggingface_hub import login as _hf_login
        _hf_login(os.environ["HF_TOKEN"], add_to_git_credential=False)
except Exception as _e:
    print(f"(secrets not configured: {_e})")

assert os.environ.get("OPENROUTER_API_KEY"), \
    "Set OPENROUTER_API_KEY in Colab Secrets."

print("Bootstrap done.")
print(display_locked_state())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
GPU: NVIDIA A100-SXM4-80GB   VRAM: 85.1 GB   torch=2.10.0+cu128


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Bootstrap done.
Locked prompts (single source of truth: src/prompts.py)
  CACHE_NAMESPACE_VERSION = 'v3'
  digests:
    TEACHER_SYSTEM_HARMFUL         sha256[:16] = ed7c66091dcb0423
    TEACHER_SYSTEM_BENIGN          sha256[:16] = 4a794d6899bf93c5
    JUDGE_SYSTEM                   sha256[:16] = 6836bdf73c5c8548
    JUDGE_USER_TEMPLATE            sha256[:16] = f065343fdbf745d4


## Configuration

In [25]:
ANCHOR  = "google/gemma-3-4b-it"
TEACHER_HARMFUL = "anthropic/claude-opus-4.7"
TEACHER_BENIGN  = "meta-llama/Llama-3.3-70B-Instruct"
JUDGE_MODEL     = "openai/gpt-5-mini"

# Cross-lingual stream
XL_TARGET           = 200
XL_SAMPLES_PER_PROMPT = 8     # n=8, same as notebook 02 Stage 1
XL_BATCH_SIZE_GEN   = 16      # base anchor sampling

# Over-refusal counter stream
OVERREF_TARGET      = 100

# Generation budgets
TEACHER_MAX_TOKENS = 1024
BENIGN_MAX_TOKENS  = 600
JUDGE_MAX_TOKENS   = 300
TEACHER_WORKERS    = 4
BENIGN_WORKERS     = 4
JUDGE_WORKERS      = 8

short  = short_of(ANCHOR)
family = family_of(ANCHOR)

PREF_OUT      = PREFS_DIR / short
PAIRS_V1_PATH = PREF_OUT / "preferences_v1.jsonl"
PAIRS_V2_PATH = PREF_OUT / "preferences_v2.jsonl"

# Per-anchor augmentation cache
AUG_OUT       = AUG_DIR / short
AUG_OUT.mkdir(parents=True, exist_ok=True)
XL_PATH       = AUG_OUT / "stage4_xl.jsonl"
OVERREF_PATH  = AUG_OUT / "stage4_overref.jsonl"

# Frozen HarmBench source
HARMBENCH_PATH = AUG_DIR / "harmbench_behaviors_v1.csv"

print(f"anchor: {ANCHOR}  short={short}  family={family}")
print(f"core pairs file: {PAIRS_V1_PATH}")
print(f"augmentation outputs: {AUG_OUT}")

anchor: google/gemma-3-4b-it  short=gemma-3-4b  family=gemma
core pairs file: /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/gemma-3-4b/preferences_v1.jsonl
augmentation outputs: /content/drive/MyDrive/PhD/paper3-alignment/data/augmentation/gemma-3-4b


## Pre-flight gate

Verifies the smoke gate is fresh (notebook 00 written within 7 days,
digests aligned) and that notebook 02 has produced a core
`preferences_v1.jsonl` for this anchor. Stage 4 will not run unless
both conditions hold.

In [26]:
print("Pre-flight gate (PAPER3_PLAN section 15.7)")
print("-" * 72)
try:
    smoke = verify_smoke_gate(PREFS_DIR, ANCHOR)
except SmokeGateNotPassed as e:
    raise SystemExit(f"\nPRE-FLIGHT FAIL\n{e}\n")

if not PAIRS_V1_PATH.exists():
    raise SystemExit(
        f"\nPRE-FLIGHT FAIL\n"
        f"Core preference file not found at {PAIRS_V1_PATH}.\n"
        f"Run notebook 02 (Stage 1+2) for {short} first."
    )

core_pairs = [json.loads(l) for l in PAIRS_V1_PATH.open()]
print(f"smoke ok            : {smoke['completed_at']}")
print(f"core pairs (v1)     : {len(core_pairs)}")
print(f"prompt digests align: yes (verified)")
print(f"\nPRE-FLIGHT PASS. Stage 4 unlocked for {short!r}.")

Pre-flight gate (PAPER3_PLAN section 15.7)
------------------------------------------------------------------------
smoke ok            : 2026-05-11T15:09:20+00:00
core pairs (v1)     : 63
prompt digests align: yes (verified)

PRE-FLIGHT PASS. Stage 4 unlocked for 'gemma-3-4b'.


## OpenRouter wrapper

In [27]:
import requests
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

OPENROUTER_URL = "https://openrouter.ai/api/v1/chat/completions"
SESSION = requests.Session()
SESSION.headers.update({
    "Authorization": f"Bearer {os.environ['OPENROUTER_API_KEY']}",
    "Content-Type":  "application/json",
    "HTTP-Referer":  "https://github.com/anonymised/rosafety-align",
    "X-Title":       "RoSafetyAlign Stage 4 augmentation",
})

@retry(stop=stop_after_attempt(4),
       wait=wait_exponential(multiplier=1, min=2, max=20),
       retry=retry_if_exception_type((requests.exceptions.RequestException, RuntimeError)))
def or_chat_completion(*, model, system, user, max_tokens, temperature=0.0,
                        response_format=None):
    payload = {
        "model": model,
        "temperature": temperature,
        "max_tokens": max_tokens,
        "messages": [
            {"role": "system", "content": system},
            {"role": "user",   "content": user},
        ],
        "usage": {"include": True},
    }
    if response_format is not None:
        payload["response_format"] = response_format
    if model.startswith("openai/gpt-5"):
        payload["reasoning"] = {"effort": "minimal"}

    r = SESSION.post(OPENROUTER_URL, json=payload, timeout=120)
    if r.status_code == 429:
        raise RuntimeError("429 rate limited")
    r.raise_for_status()
    data = r.json()
    choice = data["choices"][0]
    text   = choice["message"].get("content", "") or ""
    usage  = data.get("usage", {}) or {}
    details = usage.get("completion_tokens_details") or {}
    return {
        "text": text,
        "finish_reason": choice.get("finish_reason", ""),
        "native_finish_reason": choice.get("native_finish_reason", ""),
        "usage": {
            "prompt_tokens":     int(usage.get("prompt_tokens", 0) or 0),
            "completion_tokens": int(usage.get("completion_tokens", 0) or 0),
            "reasoning_tokens":  int(details.get("reasoning_tokens", 0) or 0),
            "cost_usd":          float(usage.get("cost", 0.0) or 0.0),
        },
    }

def parse_judge_json(text: str) -> dict:
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    start = text.find("{")
    if start == -1:
        return {"label": "compliance", "justification": "judge JSON parse failed"}
    depth = 0
    for i in range(start, len(text)):
        c = text[i]
        if c == "{": depth += 1
        elif c == "}":
            depth -= 1
            if depth == 0:
                try:    return json.loads(text[start:i+1])
                except json.JSONDecodeError: continue
    return {"label": "compliance", "justification": "judge JSON parse failed"}

## Stream 1 -- Cross-lingual harmful pairs

HarmBench is the standard EN harmful-prompt benchmark (510 prompts).
First run downloads the public CSV from the official GitHub repo and
caches it on Drive. Stratified sample of 200 prompts across the 7
standard categories.

For each EN prompt:
  1. Sample n=8 base-anchor completions on the EN prompt at temperature
     1, judge each with `gpt-5-mini`. Keep first `compliance` as the
     rejected base.
  2. Mechanically translate that EN compliance to RO via Google Translate.
  3. Query Claude Opus 4.7 (TEACHER_SYSTEM_HARMFUL) on the EN prompt;
     verify the response is judged `refusal`. Use as chosen.
  4. Emit the pair `(prompt=EN, chosen=Claude RO refusal, rejected=MT'd RO)`.

Resumable: every per-prompt outcome appended to stage4_xl.jsonl on Drive.

In [28]:
# --- HarmBench source ---
if not HARMBENCH_PATH.exists():
    print(f"downloading HarmBench behaviors -> {HARMBENCH_PATH}")
    url = ("https://raw.githubusercontent.com/centerforaisafety/HarmBench/"
           "main/data/behavior_datasets/harmbench_behaviors_text_all.csv")
    import urllib.request
    urllib.request.urlretrieve(url, HARMBENCH_PATH)
print(f"HarmBench: {HARMBENCH_PATH}  ({HARMBENCH_PATH.stat().st_size:,} bytes)")

import csv
behaviors = []
with open(HARMBENCH_PATH, newline="", encoding="utf-8") as f:
    reader = csv.DictReader(f)
    for row in reader:
        # Standard split, exclude multimodal / context-dependent rows
        if row.get("FunctionalCategory", "").strip().lower() != "standard":
            continue
        behaviors.append({
            "id":       row["BehaviorID"].strip(),
            "behavior": row["Behavior"].strip(),
            "category": row.get("SemanticCategory", "").strip().lower(),
        })
from collections import Counter
print(f"HarmBench standard rows: {len(behaviors)}")
print(f"per-category: {dict(Counter(b['category'] for b in behaviors))}")

# Stratified sample to XL_TARGET (~200), proportional to category sizes
import random
rng = random.Random(0xCAFE0021)
by_cat = {}
for b in behaviors:
    by_cat.setdefault(b["category"], []).append(b)
for v in by_cat.values(): rng.shuffle(v)

total = sum(len(v) for v in by_cat.values())
target_per_cat = {
    cat: max(1, round(XL_TARGET * len(v) / total))
    for cat, v in by_cat.items()
}
selected = []
for cat, items in by_cat.items():
    selected.extend(items[:target_per_cat[cat]])
rng.shuffle(selected)
selected = selected[:XL_TARGET]
print(f"\nselected {len(selected)} behaviors")
print(f"per-category: {dict(Counter(b['category'] for b in selected))}")

HarmBench: /content/drive/MyDrive/PhD/paper3-alignment/data/augmentation/harmbench_behaviors_v1.csv  (198,850 bytes)
HarmBench standard rows: 200
per-category: {'chemical_biological': 28, 'illegal': 58, 'misinformation_disinformation': 34, 'harmful': 21, 'harassment_bullying': 19, 'cybercrime_intrusion': 40}

selected 200 behaviors
per-category: {'misinformation_disinformation': 34, 'cybercrime_intrusion': 40, 'illegal': 58, 'harassment_bullying': 19, 'chemical_biological': 28, 'harmful': 21}


In [29]:
# --- Stage 4 stream 1: sample base anchor on EN prompt, find rejected ---
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_kwargs_for_local(family: str) -> dict:
    common = dict(torch_dtype=torch.bfloat16, device_map={"": 0})
    if family.startswith("gemma"):
        return {**common, "attn_implementation": "eager"}
    return {**common, "attn_implementation": "sdpa"}

done_keys = set()
if XL_PATH.exists():
    for line in open(XL_PATH, encoding="utf-8"):
        r = json.loads(line)
        if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
            done_keys.add(r["id"])
    print(f"resuming -- {len(done_keys)} XL prompts already processed under {CACHE_NAMESPACE_VERSION}")

todo = [b for b in selected if b["id"] not in done_keys]

if todo:
    tok = AutoTokenizer.from_pretrained(ANCHOR, padding_side="left")
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    mdl = AutoModelForCausalLM.from_pretrained(ANCHOR, **load_kwargs_for_local(family))
    mdl.eval()

    def judge_compliance_via_or(prompt, response):
        r = or_chat_completion(
            model=JUDGE_MODEL, system=JUDGE_SYSTEM,
            user=JUDGE_USER_TEMPLATE.format(
                prompt=prompt, response=response or "[empty response]"),
            max_tokens=JUDGE_MAX_TOKENS, temperature=0.0,
            response_format={"type": "json_object"},
        )
        return parse_judge_json(r["text"]).get("label") == "compliance"

    from deep_translator import GoogleTranslator
    def mt_en_to_ro(text):
        text = (text or "").strip()
        if not text:
            return ""
        # GoogleTranslator caps inputs around ~5000 chars. Truncate
        # safely; the rejected side is typically <=300 chars so this
        # rarely fires.
        return GoogleTranslator(source="en", target="ro").translate(text[:4500])

    # XL stage 1: sample rejected
    from collections import Counter as _Counter
    with open(XL_PATH, "a", encoding="utf-8") as fout:
        for chunk_start in range(0, len(todo), XL_BATCH_SIZE_GEN):
            chunk = todo[chunk_start:chunk_start + XL_BATCH_SIZE_GEN]
            prompts = [b["behavior"] for b in chunk]
            formatted = [
                tok.apply_chat_template(
                    [{"role": "user", "content": p}],
                    tokenize=False, add_generation_prompt=True,
                ) for p in prompts
            ]
            expanded = [t for t in formatted for _ in range(XL_SAMPLES_PER_PROMPT)]
            inputs = tok(expanded, return_tensors="pt", padding=True,
                         truncation=True, max_length=1024).to(mdl.device)
            with torch.no_grad():
                out = mdl.generate(
                    **inputs, max_new_tokens=256,
                    do_sample=True, temperature=1.0, top_p=0.95,
                    pad_token_id=tok.pad_token_id,
                )
            input_len = inputs["input_ids"].shape[1]
            gens = [tok.decode(seq[input_len:], skip_special_tokens=True) for seq in out]

            for j, b in enumerate(chunk):
                completions = gens[j * XL_SAMPLES_PER_PROMPT : (j + 1) * XL_SAMPLES_PER_PROMPT]
                rejected_en = None
                for ci, comp in enumerate(completions):
                    if judge_compliance_via_or(b["behavior"], comp):
                        rejected_en = {"sample_index": ci, "text": comp}
                        break
                if rejected_en is None:
                    # No compliance -> drop
                    fout.write(json.dumps({
                        "id": b["id"], "category": b["category"],
                        "behavior": b["behavior"],
                        "rejected_en": None, "rejected_ro": None,
                        "chosen": None, "chosen_judge_label": None, "ok": False,
                        "prompt_version": CACHE_NAMESPACE_VERSION,
                        "drop_reason": "no_base_compliance",
                    }, ensure_ascii=False) + "\n")
                    fout.flush(); continue

                # Translate rejected to RO
                try:
                    rejected_ro = mt_en_to_ro(rejected_en["text"])
                except Exception as e:
                    rejected_ro = ""
                    print(f"  MT failed for {b['id']}: {e}")

                # Query Claude on the EN prompt
                try:
                    chosen_resp = or_chat_completion(
                        model=TEACHER_HARMFUL, system=TEACHER_SYSTEM_HARMFUL,
                        user=b["behavior"], max_tokens=TEACHER_MAX_TOKENS,
                        temperature=0.0,
                    )
                    chosen_text = chosen_resp["text"]
                except Exception as e:
                    chosen_text = ""
                    print(f"  teacher failed for {b['id']}: {e}")

                # Judge the teacher response
                try:
                    verdict = parse_judge_json(or_chat_completion(
                        model=JUDGE_MODEL, system=JUDGE_SYSTEM,
                        user=JUDGE_USER_TEMPLATE.format(
                            prompt=b["behavior"],
                            response=chosen_text or "[empty response]"),
                        max_tokens=JUDGE_MAX_TOKENS, temperature=0.0,
                        response_format={"type": "json_object"},
                    )["text"])
                    judge_label = verdict.get("label", "compliance")
                except Exception:
                    judge_label = "error"

                ok = bool(chosen_text.strip() and judge_label == "refusal" and rejected_ro)
                fout.write(json.dumps({
                    "id": b["id"], "category": b["category"],
                    "behavior": b["behavior"],
                    "rejected_en": rejected_en, "rejected_ro": rejected_ro,
                    "chosen": chosen_text, "chosen_judge_label": judge_label,
                    "ok": ok,
                    "prompt_version": CACHE_NAMESPACE_VERSION,
                }, ensure_ascii=False) + "\n")
                fout.flush()

            if (chunk_start // XL_BATCH_SIZE_GEN) % 4 == 0:
                print(f"  XL: {chunk_start + len(chunk)}/{len(todo)} ({short})")

    del mdl, tok; gc.collect(); torch.cuda.empty_cache()

# Summary
xl_records = [json.loads(l) for l in open(XL_PATH, encoding="utf-8")
              if json.loads(l).get("prompt_version") == CACHE_NAMESPACE_VERSION]
ok = sum(1 for r in xl_records if r["ok"])
print(f"\nXL stream: {ok}/{len(xl_records)} ok pairs")
drop_reasons = Counter(r.get("drop_reason", "judge_or_teacher") for r in xl_records if not r["ok"])
print(f"drop reasons: {dict(drop_reasons)}")

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/33.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/35.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

  XL: 16/200 (gemma-3-4b)
  XL: 80/200 (gemma-3-4b)
  XL: 144/200 (gemma-3-4b)
  XL: 200/200 (gemma-3-4b)

XL stream: 42/200 ok pairs
drop reasons: {'no_base_compliance': 155, 'judge_or_teacher': 3}


## Stream 2 -- Over-refusal counter-pairs

Source: Paper 2's `overrefusal.jsonl` (100 benign Romanian prompts that
superficially look risky -- cooking dangerous mushrooms, first aid for
tick bites, civic information).

For each prompt:
  - **chosen**: Llama-3.3-70B-Instruct under TEACHER_SYSTEM_BENIGN,
    helpful answer. Verified by `gpt-5-mini` to be a `compliance`.
  - **rejected**: random sample from a 15-template hand-crafted Romanian
    apologetic-refusal pool. Templated rather than elicited because (a)
    the rejected side only needs to be recognisable as a refusal during
    DPO; (b) elicitation introduces per-anchor regeneration cost; (c)
    some prompts are borderline where an elicited refusal might be the
    correct response.

In [30]:
REFUSAL_TEMPLATES = [
    "Imi pare rau, nu te pot ajuta cu asta.",
    "Ca model de limbaj, nu pot oferi sfaturi in acest sens.",
    "Aceasta este o chestiune delicata; prefer sa nu raspund.",
    "Nu pot raspunde la aceasta intrebare. Te rog sa consulti un specialist.",
    "Ma tem ca nu pot oferi informatii despre acest subiect.",
    "Nu ma simt confortabil sa raspund la aceasta intrebare.",
    "Aceasta intrebare depaseste limitele mele. Nu pot sa raspund.",
    "Imi cer scuze, dar nu pot oferi un raspuns la aceasta intrebare.",
    "Ca asistent AI, nu sunt in masura sa raspund la astfel de intrebari.",
    "Nu pot oferi sfaturi in aceasta privinta. Te rog sa cauti ajutor de la o sursa autorizata.",
    "Prefer sa nu ma pronunt pe aceasta tema.",
    "Aceasta intrebare ar putea fi sensibila; nu pot raspunde.",
    "Nu am voie sa discut acest subiect.",
    "Imi pare rau, dar nu pot ajuta cu aceasta cerere.",
    "Ma scuzi, dar prefer sa nu intervin in aceasta chestiune.",
]

# Source benign prompts
benign_path = PAPER2_ROOT / "benchmark" / "expanded" / "overrefusal.jsonl"
benign_prompts = [json.loads(l) for l in benign_path.open(encoding="utf-8")]
print(f"benign prompts: {len(benign_prompts)} (target {OVERREF_TARGET})")

# Take a deterministic shuffle so the over-refusal stream is stable
# across re-runs (same prompts every session).
import random
rng = random.Random(0xBEEFCAFE)
benign_shuffled = list(benign_prompts)
rng.shuffle(benign_shuffled)
benign_target = benign_shuffled[:OVERREF_TARGET]

benign prompts: 100 (target 100)


In [31]:
# --- Stage 4 stream 2: helpful-answer chosen + templated refusal rejected ---
done_keys = set()
if OVERREF_PATH.exists():
    for line in open(OVERREF_PATH, encoding="utf-8"):
        r = json.loads(line)
        if r.get("prompt_version") == CACHE_NAMESPACE_VERSION:
            done_keys.add(r["id"])
    print(f"resuming -- {len(done_keys)} over-refusal prompts already processed")

todo = [p for p in benign_target if p["id"] not in done_keys]

if todo:
    rng = random.Random(0xFEEDFACE)

    def helpful_then_refusal(item):
        try:
            resp = or_chat_completion(
                model=TEACHER_BENIGN, system=TEACHER_SYSTEM_BENIGN,
                user=item["text_ro"], max_tokens=BENIGN_MAX_TOKENS,
                temperature=0.0,
            )
            helpful = resp["text"]
        except Exception as e:
            return item["id"], {"error": f"benign teacher: {e}"}
        # Verify with judge that it's a compliance, not a refusal
        try:
            verdict = parse_judge_json(or_chat_completion(
                model=JUDGE_MODEL, system=JUDGE_SYSTEM,
                user=JUDGE_USER_TEMPLATE.format(
                    prompt=item["text_ro"],
                    response=helpful or "[empty response]"),
                max_tokens=JUDGE_MAX_TOKENS, temperature=0.0,
                response_format={"type": "json_object"},
            )["text"])
            judge_label = verdict.get("label", "compliance")
        except Exception:
            judge_label = "error"
        return item["id"], {
            "helpful": helpful, "judge_label": judge_label,
        }

    helpful_results = {}
    with ThreadPoolExecutor(max_workers=BENIGN_WORKERS) as pool:
        futures = [pool.submit(helpful_then_refusal, x) for x in todo]
        for i, fut in enumerate(as_completed(futures), 1):
            item_id, resp = fut.result()
            helpful_results[item_id] = resp
            if i % 25 == 0:
                print(f"  overref helpful: {i}/{len(todo)}")

    # Emit pairs
    with open(OVERREF_PATH, "a", encoding="utf-8") as fout:
        for x in todo:
            resp = helpful_results.get(x["id"], {})
            if "error" in resp:
                fout.write(json.dumps({
                    "id": x["id"], "category": x.get("category", ""),
                    "prompt": x["text_ro"],
                    "chosen": None, "rejected": None,
                    "ok": False, "error": resp["error"],
                    "prompt_version": CACHE_NAMESPACE_VERSION,
                }, ensure_ascii=False) + "\n"); fout.flush(); continue

            helpful = resp.get("helpful") or ""
            judge_label = resp.get("judge_label", "compliance")
            # Templated refusal -- deterministic per id so re-runs are stable
            templ_idx = int(hashlib.sha256(x["id"].encode()).hexdigest(), 16) % len(REFUSAL_TEMPLATES)
            rejected = REFUSAL_TEMPLATES[templ_idx]
            # ok if helpful answer is non-empty AND judge says compliance
            ok = bool(helpful.strip() and judge_label == "compliance")
            fout.write(json.dumps({
                "id": x["id"], "category": x.get("category", ""),
                "prompt": x["text_ro"],
                "chosen": helpful,
                "chosen_judge_label": judge_label,
                "rejected": rejected,
                "rejected_template_index": templ_idx,
                "ok": ok,
                "prompt_version": CACHE_NAMESPACE_VERSION,
            }, ensure_ascii=False) + "\n")
            fout.flush()

overref_records = [json.loads(l) for l in open(OVERREF_PATH, encoding="utf-8")
                   if json.loads(l).get("prompt_version") == CACHE_NAMESPACE_VERSION]
ok = sum(1 for r in overref_records if r["ok"])
print(f"\noverref stream: {ok}/{len(overref_records)} ok pairs")

  overref helpful: 25/100
  overref helpful: 50/100
  overref helpful: 75/100
  overref helpful: 100/100

overref stream: 100/100 ok pairs


## Merge -- preferences_v2.jsonl

Combines the core Stage 1+2 pairs (`preferences_v1.jsonl`) with the two
augmentation streams into a single training file. Each pair carries a
`meta.source` field that records which stream it came from, useful for
per-stream ablations later.

In [32]:
pairs = []

# Core Stage 2
for r in core_pairs:
    pairs.append({**r, "meta": {**r.get("meta", {}), "source": "core_s2"}})

# XL stream
for r in xl_records:
    if not r.get("ok"): continue
    pairs.append({
        "prompt":   r["behavior"],
        "chosen":   r["chosen"],
        "rejected": r["rejected_ro"],
        "meta": {
            "source": "xl",
            "id": r["id"], "category": r["category"],
            "anchor": ANCHOR, "teacher": TEACHER_HARMFUL,
            "rejected_sample_index": r["rejected_en"]["sample_index"],
            "rejected_en_preview": r["rejected_en"]["text"][:160],
        },
    })

# Over-refusal stream
for r in overref_records:
    if not r.get("ok"): continue
    pairs.append({
        "prompt":   r["prompt"],
        "chosen":   r["chosen"],
        "rejected": r["rejected"],
        "meta": {
            "source": "overref",
            "id": r["id"], "category": r["category"],
            "anchor": ANCHOR, "teacher": TEACHER_BENIGN,
            "rejected_template_index": r["rejected_template_index"],
        },
    })

# Stable order: by source then id, so reruns produce identical files
pairs.sort(key=lambda p: (p.get("meta", {}).get("source", ""),
                          p.get("meta", {}).get("id", "")))

with open(PAIRS_V2_PATH, "w", encoding="utf-8") as f:
    for p in pairs: f.write(json.dumps(p, ensure_ascii=False) + "\n")

sha = hashlib.sha256(PAIRS_V2_PATH.read_bytes()).hexdigest()[:16]

by_source = Counter(p["meta"]["source"] for p in pairs)
print(f"=== preferences_v2.jsonl ===")
print(f"  total pairs: {len(pairs)}")
print(f"  by source  : {dict(by_source)}")
print(f"  sha256[:16]: {sha}")
print(f"  path       : {PAIRS_V2_PATH}")

(PREF_OUT / "preferences_v2.meta.json").write_text(json.dumps({
    "anchor": ANCHOR,
    "teacher_harmful": TEACHER_HARMFUL,
    "teacher_benign":  TEACHER_BENIGN,
    "n_pairs": len(pairs),
    "n_by_source": dict(by_source),
    "sha256_16": sha,
    "core_pairs_path": str(PAIRS_V1_PATH),
    "xl_path":         str(XL_PATH),
    "overref_path":    str(OVERREF_PATH),
    "prompt_digests":  dict(PROMPT_DIGESTS),
    "cache_namespace_version": CACHE_NAMESPACE_VERSION,
    "augmentations_applied": True,
    "built_at": datetime.now(timezone.utc).isoformat(timespec="seconds"),
}, indent=2))

=== preferences_v2.jsonl ===
  total pairs: 205
  by source  : {'core_s2': 63, 'overref': 100, 'xl': 42}
  sha256[:16]: a9ec108c1c85fab3
  path       : /content/drive/MyDrive/PhD/paper3-alignment/data/preferences/gemma-3-4b/preferences_v2.jsonl


934